In [44]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import unidecode
from pyspark.sql.types import StringType
from pyspark.sql import DataFrame

In [34]:
spark = SparkSession.builder \
            .appName("Clean location data") \
            .config("spark.sql.catalog.nessie.ref", "main") \
            .getOrCreate()

In [3]:
vietnam_provinces = [
    "An Giang",
    "Bà Rịa - Vũng Tàu",
    "Bắc Giang",
    "Bắc Kạn",
    "Bạc Liêu",
    "Bắc Ninh",
    "Bến Tre",
    "Bình Định",
    "Bình Dương",
    "Bình Phước",
    "Bình Thuận",
    "Cà Mau",
    "Cần Thơ",
    "Cao Bằng",
    "Đà Nẵng",
    "Đắk Lắk",
    "Đắk Nông",
    "Điện Biên",
    "Đồng Nai",
    "Đồng Tháp",
    "Gia Lai",
    "Hà Giang",
    "Hà Nam",
    "Hà Nội",
    "Hà Tĩnh",
    "Hải Dương",
    "Hải Phòng",
    "Hậu Giang",
    "Hòa Bình",
    "Hưng Yên",
    "Khánh Hòa",
    "Kiên Giang",
    "Kon Tum",
    "Lai Châu",
    "Lâm Đồng",
    "Lạng Sơn",
    "Lào Cai",
    "Long An",
    "Nam Định",
    "Nghệ An",
    "Ninh Bình",
    "Ninh Thuận",
    "Phú Thọ",
    "Phú Yên",
    "Quảng Bình",
    "Quảng Nam",
    "Quảng Ngãi",
    "Quảng Ninh",
    "Quảng Trị",
    "Sóc Trăng",
    "Sơn La",
    "Tây Ninh",
    "Thái Bình",
    "Thái Nguyên",
    "Thanh Hóa",
    "Thừa Thiên Huế",
    "Tiền Giang",
    "Hồ Chí Minh",
    "Trà Vinh",
    "Tuyên Quang",
    "Vĩnh Long",
    "Vĩnh Phúc",
    "Yên Bái"
]

In [4]:
unidecode_udf = F.udf(lambda x: unidecode(x) if x else None, StringType())

In [5]:
vietnam_provinces_df = spark.createDataFrame(vietnam_provinces, "string").toDF("province")

vietnam_provinces_df = (
    vietnam_provinces_df
    .withColumn("province_clean", unidecode_udf(F.col("province")))
    .withColumn("province_clean", F.regexp_replace(F.col("province_clean"), r'[^\w\s]', ''))
    .withColumn("province_clean", F.regexp_replace(F.col("province_clean"), r'\s+', ' '))
    .withColumn("province_clean", F.trim(F.col("province_clean")))
    .withColumn("province_clean", F.lower(F.col("province_clean")))
)

vietnam_provinces_df.show(truncate=False, n=63)

+-----------------+---------------+
|province         |province_clean |
+-----------------+---------------+
|An Giang         |an giang       |
|Bà Rịa - Vũng Tàu|ba ria vung tau|
|Bắc Giang        |bac giang      |
|Bắc Kạn          |bac kan        |
|Bạc Liêu         |bac lieu       |
|Bắc Ninh         |bac ninh       |
|Bến Tre          |ben tre        |
|Bình Định        |binh dinh      |
|Bình Dương       |binh duong     |
|Bình Phước       |binh phuoc     |
|Bình Thuận       |binh thuan     |
|Cà Mau           |ca mau         |
|Cần Thơ          |can tho        |
|Cao Bằng         |cao bang       |
|Đà Nẵng          |da nang        |
|Đắk Lắk          |dak lak        |
|Đắk Nông         |dak nong       |
|Điện Biên        |dien bien      |
|Đồng Nai         |dong nai       |
|Đồng Tháp        |dong thap      |
|Gia Lai          |gia lai        |
|Hà Giang         |ha giang       |
|Hà Nam           |ha nam         |
|Hà Nội           |ha noi         |
|Hà Tĩnh          |ha tinh  

In [6]:
vietnam_provinces_df.createOrReplaceTempView("provinces")

In [25]:
# Get jobs dataframe to clean location column
jobs_df = spark.read.format("iceberg").load("nessie.silver.jobs")
jobs_df.show()

+------------+--------------------+--------------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|    platform|         title_clean|      location_clean|       company_clean|min_salary|max_salary|currency|salary_type|min_years|max_years|experience_type|education_standard|work_form_standard|quantity_normalized|expired_date_norm|     industry_sector|          skills_all| category_name_final|                link|
+------------+--------------------+--------------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|       topcv|kế toán trưởng (3...|              

In [27]:
jobs_df = jobs_df.select("platform", "title_clean", "location_clean")

# Chuẩn hóa cột location_clean trong jobs_df theo cùng một cách
jobs_df = (jobs_df.withColumn("location_clean", unidecode_udf(F.col("location_clean")))
    .withColumn("location_clean", F.regexp_replace(F.col("location_clean"), r'[^\w\s]', ''))
    .withColumn("location_clean", F.regexp_replace(F.col("location_clean"), r'\s+', ' '))
    .withColumn("location_clean", F.trim(F.col("location_clean")))
    .withColumn("location_clean", F.lower(F.col("location_clean")))
)

In [28]:
jobs_df.createOrReplaceTempView("jobs")

In [31]:
# Join và lấy province nếu province_clean có chứa trong location_clean
jobs_df = spark.sql("""
    WITH joined AS (
        SELECT 
            j.*,
            p.province,
            ROW_NUMBER() OVER (
                PARTITION BY j.platform, j.title_clean, j.location_clean
                ORDER BY LENGTH(p.province_clean) DESC
            ) as rn
        FROM jobs j
        LEFT JOIN provinces p 
            ON j.location_clean LIKE CONCAT('%', p.province_clean, '%')
    )
    SELECT 
        platform,
        title_clean,
        location_clean,
        province_clean,
        province
    FROM joined
    WHERE rn = 1
""")

jobs_df.show(truncate=False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `province_clean` cannot be resolved. Did you mean one of the following? [`province`, `title_clean`, `location_clean`, `rn`, `platform`].; line 18 pos 8;
'WithCTE
:- CTERelationDef 3, false
:  +- SubqueryAlias joined
:     +- Project [platform#350, title_clean#351, location_clean#509, province#2, rn#520]
:        +- Project [platform#350, title_clean#351, location_clean#509, province#2, _w0#522, rn#520, rn#520]
:           +- Window [row_number() windowspecdefinition(platform#350, title_clean#351, location_clean#509, _w0#522 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rn#520], [platform#350, title_clean#351, location_clean#509], [_w0#522 DESC NULLS LAST]
:              +- Project [platform#350, title_clean#351, location_clean#509, province#2, length(province_clean#17) AS _w0#522]
:                 +- Join LeftOuter, location_clean#509 LIKE concat(

In [24]:
# Null rate and null sum of province column
null_rate = jobs_df.filter(F.col("province").isNull()).count() / jobs_df.count()
null_sum = jobs_df.filter(F.col("province").isNull()).count()
print(f"Null rate of province column: {null_rate:.2%}")
print(f"Null sum of province column: {null_sum}")

Null rate of province column: 9.48%
Null sum of province column: 950


In [23]:
jobs_pandas_df = jobs_df.toPandas()
jobs_pandas_df = jobs_pandas_df[jobs_pandas_df["province"].isnull()]
jobs_pandas_df.head(20)

,platform,title_clean,location_clean,province_clean,province
20,careerviet,[factory 1] qa officer (pharmacy),"16 vsip, street 5, vietnam-singapore industria...",16 vsip street 5 vietnamsingapore industrial park,None
21,careerviet,[factory 1] techniques engineer (automation/ e...,"16 vsip, street 5, vietnam-singapore industria...",16 vsip street 5 vietnamsingapore industrial park,None
26,careerviet,[hcm - bình thạnh] cửa hàng phó,"24 nguyễn gia trí, phường 25, quận bình thạnh,...",24 nguyen gia tri phuong 25 quan binh thanh tphcm,None
27,careerviet,[hcm] business intelligence analyst - chuyên v...,"floor 17, lim 2 tower, 62a cach mang thang tam...",floor 17 lim 2 tower 62a cach mang thang tam w...,None
34,careerviet,[hà nội] chuyên viên kinh doanh bảo hiểm phi n...,"tầng 10 tháp đông, tòa nhà lotte, 54 liễu giai...",tang 10 thap dong toa nha lotte 54 lieu giai q...,None
35,careerviet,[hà nội] chuyên viên tư vấn bảo hiểm,"tầng 10 tháp đông, tòa nhà lotte, 54 liễu giai...",tang 10 thap dong toa nha lotte 54 lieu giai q...,None
47,careerviet,[hà nội] trưởng nhóm business analyst,"tầng 7, tòa leadvisors, 643 phạm văn đồng",tang 7 toa leadvisors 643 pham van dong,None
50,careerviet,[hồ chí minh] chuyên gia/chuyên viên cao cấp x...,phan đình giót,phan dinh giot,None
51,careerviet,[i1] admin bộ phận phát triển kinh doanh (hn),"tầng 3, tòa nhà trung yên 1, số 1a vũ phạm hàm",tang 3 toa nha trung yen 1 so 1a vu pham ham,None
63,careerviet,[novadreams] chuyên viên cao cấp thu hút nhân tài,"tòa nhà 2 bis nguyễn thị minh khai, quận 1",toa nha 2 bis nguyen thi minh khai quan 1,None


In [54]:
def normalize_location(jobs_df: DataFrame, spark: SparkSession) -> DataFrame:
    """ Chuẩn hóa location, đưa về tên tỉnh thành chuẩn của Việt Nam nếu có thể"""
    
    vietnam_provinces = [
        "An Giang",
        "Bà Rịa - Vũng Tàu",
        "Bắc Giang",
        "Bắc Kạn",
        "Bạc Liêu",
        "Bắc Ninh",
        "Bến Tre",
        "Bình Định",
        "Bình Dương",
        "Bình Phước",
        "Bình Thuận",
        "Cà Mau",
        "Cần Thơ",
        "Cao Bằng",
        "Đà Nẵng",
        "Đắk Lắk",
        "Đắk Nông",
        "Điện Biên",
        "Đồng Nai",
        "Đồng Tháp",
        "Gia Lai",
        "Hà Giang",
        "Hà Nam",
        "Hà Nội",
        "Hà Tĩnh",
        "Hải Dương",
        "Hải Phòng",
        "Hậu Giang",
        "Hòa Bình",
        "Hưng Yên",
        "Khánh Hòa",
        "Kiên Giang",
        "Kon Tum",
        "Lai Châu",
        "Lâm Đồng",
        "Lạng Sơn",
        "Lào Cai",
        "Long An",
        "Nam Định",
        "Nghệ An",
        "Ninh Bình",
        "Ninh Thuận",
        "Phú Thọ",
        "Phú Yên",
        "Quảng Bình",
        "Quảng Nam",
        "Quảng Ngãi",
        "Quảng Ninh",
        "Quảng Trị",
        "Sóc Trăng",
        "Sơn La",
        "Tây Ninh",
        "Thái Bình",
        "Thái Nguyên",
        "Thanh Hóa",
        "Thừa Thiên Huế",
        "Tiền Giang",
        "Hồ Chí Minh",
        "Trà Vinh",
        "Tuyên Quang",
        "Vĩnh Long",
        "Vĩnh Phúc",
        "Yên Bái"
    ]

    unidecode_udf = F.udf(lambda x: unidecode.unidecode(x) if x else None, StringType())

    # Tạo DataFrame chứa tên tỉnh thành đã chuẩn hóa và phiên bản unidecode của nó
    vietnam_provinces_df = spark.createDataFrame(vietnam_provinces, "string").toDF("province")

    vietnam_provinces_df = (
        vietnam_provinces_df
        .withColumn("province_clean", unidecode_udf(F.col("province")))
        .withColumn("province_clean", F.regexp_replace(F.col("province_clean"), r'[^\w\s]', ''))
        .withColumn("province_clean", F.regexp_replace(F.col("province_clean"), r'\s+', ' '))
        .withColumn("province_clean", F.trim(F.col("province_clean")))
        .withColumn("province_clean", F.lower(F.col("province_clean")))
    )

    # Chuẩn hóa cột location_clean trong jobs_df theo cùng một cách
    jobs_norm_df = (jobs_df.withColumn("location_clean", unidecode_udf(F.col("location_clean")))
        .withColumn("location_clean", F.regexp_replace(F.col("location_clean"), r'[^\w\s]', ''))
        .withColumn("location_clean", F.regexp_replace(F.col("location_clean"), r'\s+', ' '))
        .withColumn("location_clean", F.trim(F.col("location_clean")))
        .withColumn("location_clean", F.lower(F.col("location_clean")))
    )

    # Tạo view tạm thời để join với bảng tỉnh thành
    vietnam_provinces_df.createOrReplaceTempView("provinces")
    jobs_norm_df.createOrReplaceTempView("jobs")

    # Join để gán tên tỉnh thành chuẩn vào jobs_df
    jobs_norm_df = spark.sql("""
        WITH joined AS (
            SELECT 
                j.*,
                p.province,
                ROW_NUMBER() OVER (
                    PARTITION BY j.platform, j.title_clean, j.location_clean
                    ORDER BY LENGTH(p.province_clean) DESC
                ) as rn
            FROM jobs j
            LEFT JOIN provinces p 
                ON j.location_clean LIKE CONCAT('%', p.province_clean, '%')
        )
        SELECT 
            platform,
            title_clean,
            COALESCE(province, "Khác") as location_clean,
            company_clean,
            min_salary,
            max_salary,
            currency,
            salary_type,
            min_years,
            max_years,
            experience_type,
            education_standard,
            work_form_standard,
            quantity_normalized,
            expired_date_norm,
            industry_sector,
            skills_all,
            category_name_final,
            link
        FROM joined
        WHERE rn = 1
    """)
    
    # For debug: show some examples of location normalization
    print("=== Location Normalization Examples ===")
    example_locations = jobs_df.select("location_clean").distinct().limit(20)
    example_locations.show(truncate=False)

    return jobs_norm_df

In [55]:
jobs_df = spark.table("nessie.silver.jobs")
jobs_df.show()

+------------+--------------------+--------------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|    platform|         title_clean|      location_clean|       company_clean|min_salary|max_salary|currency|salary_type|min_years|max_years|experience_type|education_standard|work_form_standard|quantity_normalized|expired_date_norm|     industry_sector|          skills_all| category_name_final|                link|
+------------+--------------------+--------------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|       topcv|kế toán trưởng (3...|              

In [57]:
jobs_norm = normalize_location(jobs_df, spark)
jobs_norm.show(truncate=False)

=== Location Normalization Examples ===
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|location_clean                                                                                                                                                                    |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|238 đ. 3 tháng 2, phường 12, quận 10, hồ chí minh                                                                                                                                 |
|tòa nhà mitalab, 76 giảng võ, phường giảng võ, hà nội                                                                                                                             |
|trường emasi nam long                                 

In [53]:
# Statistics location_clean column
# Sum by location_clean
location_counts = jobs_norm.groupBy("location_clean").count().orderBy(F.desc("count"))
location_counts.show(truncate=False, n=100)

+-----------------+-----+
|location_clean   |count|
+-----------------+-----+
|Hồ Chí Minh      |3794 |
|Hà Nội           |3638 |
|Khác             |953  |
|Bắc Ninh         |197  |
|Hải Phòng        |187  |
|Hưng Yên         |167  |
|Bình Dương       |165  |
|Đồng Nai         |138  |
|Đà Nẵng          |127  |
|Tây Ninh         |71   |
|Hải Dương        |64   |
|Ninh Bình        |62   |
|Quảng Ninh       |47   |
|Long An          |46   |
|Thái Nguyên      |43   |
|Khánh Hòa        |42   |
|Cần Thơ          |38   |
|An Giang         |34   |
|Nghệ An          |34   |
|Phú Thọ          |34   |
|Bắc Giang        |29   |
|Lâm Đồng         |27   |
|Quảng Ngãi       |26   |
|Hà Nam           |25   |
|Vĩnh Phúc        |25   |
|Bà Rịa - Vũng Tàu|24   |
|Thanh Hóa        |23   |
|Điện Biên        |22   |
|Bình Định        |18   |
|Hà Tĩnh          |16   |
|Lào Cai          |16   |
|Thái Bình        |15   |
|Tuyên Quang      |15   |
|Đồng Tháp        |15   |
|Kiên Giang       |14   |
|Gia Lai    

In [58]:
# overwrite back to silver layer
jobs_norm.write.format("iceberg").mode("overwrite").save("nessie.silver.jobs")